# Parameter modification and studies

Every physical parameter in a Nefes network has a **name and a dotted address**: `"inlet.mdot"`, `"orifice.throat_area"`, `"e0.area"`, `"p_ref"`.
One generic machinery reads and writes them all -- `parameters()`, `get`, `set`, `update`, `with_params` -- with fail-closed validation, so parameter studies and sweeps are routine on any loaded case.

This notebook demonstrates, on a small non-reacting network:

1. the **inventory** (`net.parameters()`) and single reads (`get`);
2. validated in-place writes (`set` / `update`), including a composite knob and the constant-area fan-out;
3. the functional **`with_params`** idiom that keeps a loaded base pristine;
4. the warm-start-chained sweep driver **`nefes.parameter_study`** in 1-D and on a 2-D grid.

See `docs/reference/parameters.md` for the full guide.

In [ ]:
import numpy as np
import plotly.graph_objects as go

import nefes
from nefes.elements import catalog as cat
from nefes.plotting import use_nefes_theme

use_nefes_theme()

## A small network with named elements

A feed line with an orifice plate: mass-flow inlet, orifice composite, a duct, and a back-pressure outlet.
The element and edge **names** given here become the address heads (a factory-default name would be auto-numbered, `duct` -> `duct-1`, so we name what we intend to address).

In [ ]:
AREA = 5.0e-3  # line flow area [m^2]

net = nefes.Network()
n_in = net.add(cat.mass_flow_inlet(0.30, 700.0, name="feed"))
n_or = net.add(cat.orifice(1.0e-3, name="plate"))
n_du = net.add(cat.duct(length=0.5, name="line"))
n_out = net.add(cat.pressure_outlet(101325.0, name="exit"))
net.connect(n_in, n_or, AREA, name="e0")
net.connect(n_or, n_du, AREA, name="e1")
net.connect(n_du, n_out, AREA, name="e2")

net.parameters()  # the full addressable inventory: value, unit, bounds

## Reads and validated writes

`get` reads one address; `set` writes named parameters on one element; `update` batches writes by address.
Every write validates against the element's declared schema **before** anything is stored, and an unknown address raises with near-match suggestions -- never a silent no-op.

Two write paths worth noting:

- the orifice is a **composite**, so setting its `throat_area` re-runs the orifice factory and regenerates the internal edges consistently (they are never patched);
- the duct is **constant-area**, so `set("line", area=...)` fans out to both of its incident edges.

In [ ]:
print("feed.mdot           :", net.get("feed.mdot"))
print("plate.throat_area   :", net.get("plate.throat_area"))

net.set("feed", mdot=0.35, Tt=720.0)  # validated in-place write
net.update({"plate.throat_area": 1.2e-3, "line.area": 4.5e-3})  # batch, composite rebuild + area fan-out
print("after update        :", net.get("plate.throat_area"), net.get("e1.area"), net.get("e2.area"))

for bad in (lambda: net.set("feed", mdot=-0.1), lambda: net.get("plate.troat_area")):
    try:
        bad()
    except (ValueError, KeyError) as err:
        print("fail-closed         :", err)

## The functional idiom: `with_params`

For studies, prefer `with_params`: it returns a modified **deep copy**, so the base stays pristine, no state accumulates across sweep points, and each point solves independently.
Parameter writes never touch topology (edge count and order are preserved by construction), so a previous solution remains a valid **warm start** for the modified copy.

In [ ]:
base = net.copy()
sol0 = base.solve()

point = base.with_params({"feed.mdot": 0.4})
warm = point.solve(x0=sol0.x)  # warm-started from the base's converged state
print(f"base pristine: feed.mdot = {base.get('feed.mdot')}")
print(f"warm start   : converged = {warm.converged} in {warm.iterations} iterations")

## 1-D sweep: `nefes.parameter_study`

The driver packages the loop: one `with_params` copy per point, warm starts chained through `solve(x0=prev.x)`, and scalar `probe` outputs collected into grid-shaped arrays.
Here the inflow is swept and the orifice pressure drop and peak Mach number are probed.

In [ ]:
mdots = np.linspace(0.18, 0.40, 12)
res = nefes.parameter_study(
    base,
    {"feed.mdot": mdots},
    probe=lambda sol: {
        "dp": float(sol.field("p_t")[0] - sol.field("p_t")[-1]),
        "M_max": float(sol.field("M").max()),
    },
    keep_solutions=False,
)
print(res)

fig = go.Figure()
fig.add_scatter(x=mdots, y=res.probes["dp"] / 1e3, mode="lines+markers", name="orifice loss")
fig.update_xaxes(title_text="inflow mass rate [kg/s]")
fig.update_yaxes(title_text="total-pressure drop [kPa]")
fig.update_layout(title="Operating line: loss across the plate vs inflow")
fig.show()

## 2-D grid: throat area vs inflow

Multiple addresses sweep their **outer product** (`mode="grid"`; `mode="zip"` aligns equal-length lists into a path instead).
The probes come back shaped `(n_area, n_mdot)`, ready to plot as a map.
The tightest throat cannot pass the largest inflow subsonically, so `on_fail="continue"` records that corner as non-converged (`NaN` in the probes) instead of aborting the sweep.

In [ ]:
areas = np.linspace(0.9e-3, 1.6e-3, 8)
grid = nefes.parameter_study(
    base,
    {"plate.throat_area": areas, "feed.mdot": mdots},
    probe=lambda sol: {"dp": float(sol.field("p_t")[0] - sol.field("p_t")[-1])},
    keep_solutions=False,
    on_fail="continue",  # the tight-throat / high-inflow corner may choke; record NaN and march on
)
print(f"{int(grid.converged.sum())}/{grid.n_points} points converged")

fig = go.Figure(
    go.Contour(
        x=mdots, y=areas * 1e3, z=grid.probes["dp"] / 1e3,
        colorbar=dict(title="dp_t [kPa]"),
    )
)
fig.update_xaxes(title_text="inflow mass rate [kg/s]")
fig.update_yaxes(title_text="throat area [mm^2]")
fig.update_layout(title="Total-pressure drop over the (throat area, inflow) grid")
fig.show()

## Beyond the mean flow

The same base plugs into the modal continuation drivers through `Network.builder`, which returns the `build(p)` closure `eigenvalue_trajectory` and the Nyquist driver take:

```python
traj = eigenvalue_trajectory(
    base.builder("flame.Qdot"), np.linspace(1e3, 5e3, 21),
    freq_band=(50.0, 400.0), param_name="Qdot",
)
```

Object-valued fields go through the same door: `net.set("exit", perturbation_bc=PerturbationBC.open_end())`, `net.set_dynamic_source("flame", n_tau_flame(...))`, or, on a reacting case, `net.set("injector", composition={"CH4": 1.0}, basis="mole")` -- the reacting composition is validated against the loaded species library at write time.

The gas model itself (species slate, mechanism, thermo model) is deliberately **not** a parameter: changing it reshapes the problem and invalidates warm starts, so it stays behind the explicit `Network(gas=...)` construction path.